# RL Trade Visualization Notebook

Visualize trade entries and exits over candlestick data for any episode produced by the RL trainer. This notebook:

- Loads config (`configs/rl_v2/rl_config.yaml`) to find the OHLCV data path.
- Scans `logs/rl_traces/` for episode trace CSVs saved by `RLTrainerV2`.
- Lets you select an episode and plots entries/exits on a candlestick chart.
- Supports mplfinance (default) and optional Plotly for interactive charts.
- Saves enriched trades and figures to `scripts/outputs/<episode_id>/`.


In [1]:
# 1) Setup Paths and Load Config
%reload_ext autoreload
%autoreload 2

import os, sys, json, glob
from pathlib import Path
from typing import Optional, Tuple

import yaml
import pandas as pd

# Resolve project root robustly: one level up from scripts folder
# Prefer __file__ when available, otherwise infer from CWD and known repo structure
if '__file__' in globals():
    NOTEBOOK_PATH = Path(__file__).resolve()
    # If this notebook is in .../scripts/rl_trade_viz.ipynb, root is one level up
    ROOT = NOTEBOOK_PATH.parent.parent
else:
    cwd = Path(os.getcwd()).resolve()
    # If running from repo root, ROOT is cwd; if running from scripts, ROOT is parent
    if (cwd / 'configs' / 'rl_v2').exists():
        ROOT = cwd
    elif cwd.name == 'scripts' and (cwd.parent / 'configs' / 'rl_v2').exists():
        ROOT = cwd.parent
    else:
        # Fallback: climb up to find configs/rl_v2
        probe = cwd
        found = False
        for _ in range(4):
            if (probe / 'configs' / 'rl_v2').exists():
                ROOT = probe
                found = True
                break
            probe = probe.parent
        if not found:
            # Last resort: assume parent of cwd
            ROOT = cwd.parent

CONFIG_PATH = ROOT / 'configs' / 'rl_v2' / 'rl_config.yaml'
TRACES_DIR = ROOT / 'logs' / 'rl_traces'
OUTPUTS_DIR = ROOT / 'scripts' / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Allow env overrides
CONFIG_PATH = Path(os.environ.get('RL_VIZ_CONFIG', str(CONFIG_PATH)))
TRACES_DIR = Path(os.environ.get('RL_VIZ_TRACES', str(TRACES_DIR)))

assert CONFIG_PATH.exists(), f"Config not found: {CONFIG_PATH}"
assert TRACES_DIR.exists(), f"Traces directory not found: {TRACES_DIR}"

with open(CONFIG_PATH, 'r') as f:
    CFG = yaml.safe_load(f)

DATA_PATH = CFG.get('data', {}).get('path', 'data/candle_data_list_frxEURUSD_3600s.pkl')
DATA_PATH = Path(DATA_PATH) if os.path.isabs(DATA_PATH) else ROOT / DATA_PATH
print(f"Using data: {DATA_PATH}")


Using data: /Users/dirkiemyburgh/Documents/Courses/Binary_trading_classifier/data/candle_data_list_frxEURUSD_3600s.pkl


In [2]:
# 2) Install and Import Dependencies
import sys, os

# Ensure interactive widgets are available (for notebook usage)
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception:
    get_ipython().system('pip -q install ipywidgets')
    import ipywidgets as widgets
    from IPython.display import display, clear_output

import numpy as np

# Prefer Plotly as the default interactive backend
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
    # Set a renderer that works well in VS Code or notebooks
    pio.renderers.default = 'vscode' if 'VSCODE_PID' in os.environ else 'notebook_connected'
except Exception:
    get_ipython().system('pip -q install plotly')
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
    pio.renderers.default = 'vscode' if 'VSCODE_PID' in os.environ else 'notebook_connected'


In [3]:
# 3) Discover Episodes and Trace Files

def list_episode_traces(traces_dir: Path) -> pd.DataFrame:
    rows = []
    for fp in sorted(traces_dir.glob('episode_*_trace.csv')):
        name = fp.stem  # episode_001_trace
        try:
            ep_str = name.split('_')[1]
            ep = int(ep_str)
        except Exception:
            ep = None
        try:
            df = pd.read_csv(fp)
            start_ts = pd.to_datetime(df.get('date')).dropna().min()
            end_ts = pd.to_datetime(df.get('date')).dropna().max()
            trades = int((df.get('action') == 'CLOSE_LONG').sum() + (df.get('action') == 'CLOSE_SHORT').sum()) if 'action' in df else 0
        except Exception:
            start_ts, end_ts, trades = None, None, 0
        rows.append({'episode': ep, 'path': str(fp), 'start': start_ts, 'end': end_ts, 'trades': trades})
    return pd.DataFrame(rows).sort_values(['episode']).reset_index(drop=True)

EP_IDX = list_episode_traces(TRACES_DIR)
print(f"Found {len(EP_IDX)} episode traces")
EP_IDX.head(3)


Found 27 episode traces


,episode,path,start,end,trades
0,1,/Users/dirkiemyburgh/Documents/Courses/Binary_...,2020-01-14 02:00:00+00:00,2020-10-30 09:00:00+00:00,261
1,2,/Users/dirkiemyburgh/Documents/Courses/Binary_...,2020-01-14 02:00:00+00:00,2020-10-30 09:00:00+00:00,305
2,3,/Users/dirkiemyburgh/Documents/Courses/Binary_...,2020-01-14 02:00:00+00:00,2020-10-30 09:00:00+00:00,289


In [4]:
# 4) Load Candlestick Data and Compute Indicators

def load_candles(data_path: Path) -> pd.DataFrame:
    if not data_path.exists():
        raise FileNotFoundError(f"OHLCV not found: {data_path}")
    low = data_path.name.lower()
    if low.endswith('.csv'):
        df = pd.read_csv(data_path)
    elif low.endswith('.pkl') or low.endswith('.pickle'):
        obj = pd.read_pickle(data_path)
        if isinstance(obj, list):
            df = pd.DataFrame(obj)
        elif isinstance(obj, pd.DataFrame):
            df = obj.copy()
        elif isinstance(obj, dict) and 'data' in obj:
            df = pd.DataFrame(obj['data'])
        else:
            raise ValueError('Unsupported pickle format')
    else:
        raise ValueError('Unsupported OHLCV file format')

    # Normalize columns
    cols = {c.lower(): c for c in df.columns}
    def pick(name):
        return cols.get(name, name.capitalize() if name.capitalize() in df.columns else name.upper())
    out = pd.DataFrame({
        'Date': pd.to_datetime(df[pick('date')], utc=True, errors='coerce'),
        'Open': pd.to_numeric(df[pick('open')], errors='coerce'),
        'High': pd.to_numeric(df[pick('high')], errors='coerce'),
        'Low': pd.to_numeric(df[pick('low')], errors='coerce'),
        'Close': pd.to_numeric(df[pick('close')], errors='coerce'),
        'Volume': pd.to_numeric(df[pick('volume')], errors='coerce') if pick('volume') in df else 0.0,
    }).dropna()
    out = out.sort_values('Date').reset_index(drop=True)
    out = out.set_index('Date')
    return out


def parse_indicator_names(cfg: dict):
    # Support both list and dict-with-names
    inds_cfg = cfg.get('indicators', [])
    if isinstance(inds_cfg, dict):
        names = inds_cfg.get('names', [])
    else:
        names = inds_cfg
    # Normalize names: uppercase, replace hyphens/spaces, strip
    norm = []
    for n in names if isinstance(names, (list, tuple)) else []:
        if not isinstance(n, str):
            continue
        s = n.upper().replace('-', '_').replace(' ', '')
        norm.append(s)
    return set(norm)


def compute_indicators(candles: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    df = candles.copy()
    inds = parse_indicator_names(cfg)
    # Moving averages
    if 'MA20' in inds:
        df['MA20'] = df['Close'].rolling(20, min_periods=1).mean()
    if 'MA200' in inds:
        df['MA200'] = df['Close'].rolling(200, min_periods=1).mean()
    # Derivatives (require MA20 present, compute on the fly if needed)
    if any(x in inds for x in ['MA20_D1','MA20_D2','MA20_D3']) and 'MA20' not in df:
        df['MA20'] = df['Close'].rolling(20, min_periods=1).mean()
    if 'MA20_D1' in inds:
        df['MA20_D1'] = df['MA20'].diff()
    if 'MA20_D2' in inds:
        df['MA20_D2'] = df['MA20'].diff().diff()
    if 'MA20_D3' in inds:
        df['MA20_D3'] = df['MA20'].diff().diff().diff()
    # MACD
    if any(x in inds for x in ['MACD','MACD_SIGNAL','MACD_HIST']):
        ema12 = df['Close'].ewm(span=12, adjust=False).mean()
        ema26 = df['Close'].ewm(span=26, adjust=False).mean()
        macd = ema12 - ema26
        signal = macd.ewm(span=9, adjust=False).mean()
        hist = macd - signal
        if 'MACD' in inds:
            df['MACD'] = macd
        if 'MACD_SIGNAL' in inds:
            df['MACD_SIGNAL'] = signal
        if 'MACD_HIST' in inds:
            df['MACD_HIST'] = hist
    # Returns
    if 'RETURN' in inds or 'RET' in inds or 'RETURNS' in inds:
        df['Return'] = df['Close'].pct_change().fillna(0.0)
    return df

# Normalization helpers

def normalize_series(s: pd.Series, method: str, eps: float = 1e-12) -> pd.Series:
    if method is None or method.lower() in ('', 'none'):
        return s
    m = method.lower()
    if m == 'zscore':
        mu = s.mean()
        sd = s.std(ddof=0)
        return (s - mu) / (sd + eps)
    if m == 'minmax':
        mn = s.min()
        mx = s.max()
        return (s - mn) / (mx - mn + eps)
    if m == 'robust':
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        return (s - s.median()) / (iqr + eps)
    return s


def normalize_dataframe(df: pd.DataFrame, columns: list, method: str) -> pd.DataFrame:
    out = df.copy()
    for c in columns:
        if c in out:
            out[c] = normalize_series(out[c], method)
    return out

CANDLES = load_candles(DATA_PATH)
CANDLES = compute_indicators(CANDLES, CFG)
CANDLES.head(3)


,Open,High,Low,Close,Volume,MA20,MA200,MACD,MACD_SIGNAL,MACD_HIST
Date,,,,,,,,,,
2020-01-01 17:00:00+00:00,1.12120,1.12166,1.12106,1.12143,0.0,1.121430,1.121430,0.000000,0.000000,0.000000
2020-01-01 18:00:00+00:00,1.12143,1.12218,1.12142,1.12188,0.0,1.121655,1.121655,0.000036,0.000007,0.000029
2020-01-01 19:00:00+00:00,1.12188,1.12190,1.12157,1.12183,0.0,1.121713,1.121713,0.000060,0.000018,0.000042


In [5]:
# 5) Parse and Normalize RL Traces

def load_episode_trace(episode: int, ep_idx: pd.DataFrame) -> pd.DataFrame:
    row = ep_idx[ep_idx['episode'] == episode]
    if row.empty:
        raise ValueError(f"Episode {episode} not found in traces")
    path = Path(row.iloc[0]['path'])
    df = pd.read_csv(path)
    if 'date' in df:
        df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')
    else:
        # fall back to env_idx if date is missing
        df['date'] = pd.NaT
    # Normalize action/price/qty
    df['action'] = df['action'].astype(str)
    # Derive execution price from candles (we’ll align later); if environment has next open execution semantics you can use Close at env_idx
    # Here we just keep placeholders; plotting will use candle prices.
    df['qty'] = 1.0
    # Assign trade_id incrementally on each close event; opens share previous pending id
    trade_id = 0
    ids = []
    open_pending = False
    for _, r in df.iterrows():
        if r['action'] in ('OPEN_LONG', 'OPEN_SHORT'):
            trade_id += 1
            open_pending = True
            ids.append(trade_id)
        elif r['action'] in ('CLOSE_LONG', 'CLOSE_SHORT') and open_pending:
            ids.append(trade_id)
            open_pending = False
        else:
            ids.append(np.nan)
    df['trade_id'] = ids
    return df


In [6]:
# 6) Align Trades to Candles

def align_trades_to_candles(tr: pd.DataFrame, candles: pd.DataFrame, tolerance='2H') -> pd.DataFrame:
    # If trace has valid dates, merge_asof to nearest candle
    if tr['date'].notna().any():
        left = tr.copy().sort_values('date')
        right = candles.reset_index().rename(columns={'Date': 'Date'})
        aligned = pd.merge_asof(
            left.sort_values('date'),
            right.rename(columns={'Date': 'candle_time'}).sort_values('candle_time'),
            left_on='date', right_on='candle_time', direction='nearest', tolerance=pd.Timedelta(tolerance)
        )
        # Use candle close for plotting position marker y if price absent
        aligned['exec_time'] = aligned['candle_time']
        aligned['exec_price'] = aligned['Close']
    else:
        # Without dates, approximate by env_idx to candle index
        left = tr.copy().sort_values('env_idx')
        right = candles.reset_index().reset_index().rename(columns={'index': 'env_idx'})
        aligned = left.merge(right, on='env_idx', how='left')
        aligned['exec_time'] = aligned['Date']
        aligned['exec_price'] = aligned['Close']
    return aligned


In [7]:
# 7) Compute Trade Metrics

def compute_trade_metrics(aligned: pd.DataFrame) -> pd.DataFrame:
    tr = aligned.copy()
    tr = tr.sort_values(['trade_id', 'exec_time'])
    # Identify entries/exits
    tr['is_entry'] = tr['action'].isin(['OPEN_LONG', 'OPEN_SHORT'])
    tr['is_exit'] = tr['action'].isin(['CLOSE_LONG', 'CLOSE_SHORT'])

    # Pair by trade_id: first entry then next exit
    entries = tr[tr['is_entry']].groupby('trade_id').first().rename(columns={'exec_time': 'entry_time', 'exec_price': 'entry_price'})
    exits = tr[tr['is_exit']].groupby('trade_id').first().rename(columns={'exec_time': 'exit_time', 'exec_price': 'exit_price'})
    pairs = entries[['entry_time', 'entry_price']].join(exits[['exit_time', 'exit_price']], how='inner')

    # Determine side from entry action
    side_map = tr[tr['is_entry']].groupby('trade_id')['action'].first().map({'OPEN_LONG': 1, 'OPEN_SHORT': -1})
    pairs['side'] = side_map
    pairs['qty'] = 1.0
    pairs['holding_time'] = (pairs['exit_time'] - pairs['entry_time']).dt.total_seconds() / 3600.0
    pairs['pnl'] = pairs['side'] * (pairs['exit_price'] - pairs['entry_price']) * pairs['qty']

    return pairs.reset_index()


In [8]:
# 8) Interactive Episode Selector (Plotly only)
from ipywidgets import Dropdown, FloatSlider, RadioButtons, Button, HBox, VBox, Output, IntSlider

EPISODES = [int(e) for e in EP_IDX['episode'].dropna().tolist()]

episode_dd = Dropdown(options=EPISODES, description='Episode:')
tolerance_sl = FloatSlider(value=2.0, min=0.0, max=24.0, step=0.5, description='Tol (h):')
# Plotly only now
backend_rb = RadioButtons(options=['plotly'], value='plotly', description='Backend:')
max_candles_sl = IntSlider(value=5000, min=500, max=20000, step=500, description='Max candles:')
save_html_rb = RadioButtons(options=['no','yes'], value='no', description='Save HTML:')

# Normalization controls
norm_choices = ['none','zscore','minmax','robust']
norm_ohlc_dd = Dropdown(options=norm_choices, value='none', description='Norm OHLC:')
norm_ind_dd = Dropdown(options=norm_choices, value='none', description='Norm Ind:')

render_btn = Button(description='Render', button_style='success')
out = Output(layout={'border': '1px solid #ddd'})


def on_render(_):
    out.clear_output(wait=True)
    ep = int(episode_dd.value)
    tol_hours = float(tolerance_sl.value)
    tol = f"{tol_hours}H"
    with out:
        print(f"Rendering episode {ep} with tolerance {tol}...")
        tr_raw = load_episode_trace(ep, EP_IDX)
        aligned = align_trades_to_candles(tr_raw, CANDLES, tolerance=tol)
        metrics = compute_trade_metrics(aligned)
        # Limit to max_candles window if needed
        if aligned['exec_time'].notna().any():
            tmin, tmax = aligned['exec_time'].min(), aligned['exec_time'].max()
            pad = pd.Timedelta('72H')
            df = CANDLES.loc[str(tmin - pad): str(tmax + pad)]
            if len(df) > max_candles_sl.value:
                step = int(np.ceil(len(df) / max_candles_sl.value))
                df = df.iloc[::step]
        else:
            df = CANDLES.tail(max_candles_sl.value)
        fig = plot_plotly(
            aligned, df, ep,
            norm_ohlc=norm_ohlc_dd.value,
            norm_indicators=norm_ind_dd.value
        )
        if fig is not None:
            fig.show()
        save_outputs(aligned, metrics, ep, fig=fig, save_html=(save_html_rb.value=='yes'))

render_btn.on_click(on_render)

HBox([
    episode_dd, tolerance_sl, backend_rb, max_candles_sl, save_html_rb,
    norm_ohlc_dd, norm_ind_dd, render_btn
])
display(out)


Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [14]:
# 10) Plot with Plotly (with indicators)
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
except Exception:
    go = None

DEFAULT_PANEL_WEIGHTS = {
    'price': 0.55,
    'macd': 0.25,
    'volume': 0.20,
    'returns': 0.20,
}


def plot_plotly(
    aligned: pd.DataFrame,
    candles: pd.DataFrame,
    episode: int,
    panel_weights: dict = None,
    hide_volume_if_zero: bool = True,
    norm_ohlc: str = 'none',
    norm_indicators: str = 'none'
) -> "go.Figure":
    if go is None:
        print('Plotly not installed.')
        return None

    # Limit the window to keep rendering fast
    if aligned['exec_time'].notna().any():
        tmin, tmax = aligned['exec_time'].min(), aligned['exec_time'].max()
        if pd.isna(tmin) or pd.isna(tmax):
            df = candles
        else:
            pad = pd.Timedelta('72H')
            df = candles.loc[str(tmin - pad): str(tmax + pad)]
    else:
        # Fallback: recent window of candles
        df = candles.tail(5000)

    # Use normalized indicator names from config
    inds = parse_indicator_names(CFG)

    # Optionally normalize OHLC for plotting in a shape-preserving way (use Close params for all O/H/L/C)
    norm_ohlc = (norm_ohlc or 'none').lower()
    if norm_ohlc not in ('none', ''):
        ohlc_cols = ['Open','High','Low','Close']
        existing = [c for c in ohlc_cols if c in df]
        if 'Close' in df:
            if norm_ohlc == 'zscore':
                mu = float(df['Close'].mean())
                sigma = float(df['Close'].std(ddof=0)) or 1.0
                for c in existing:
                    df[c] = (df[c] - mu) / sigma
            elif norm_ohlc == 'minmax':
                vmin = float(df['Close'].min())
                vmax = float(df['Close'].max())
                denom = (vmax - vmin) or 1.0
                for c in existing:
                    df[c] = (df[c] - vmin) / denom
            elif norm_ohlc == 'robust':
                q1 = float(df['Close'].quantile(0.25))
                q3 = float(df['Close'].quantile(0.75))
                iqr = (q3 - q1) or 1.0
                med = float(df['Close'].median())
                for c in existing:
                    df[c] = (df[c] - med) / iqr

    # Optionally normalize indicators for plotting (MA*, MACD*, Return, derivatives)
    norm_indicators = (norm_indicators or 'none').lower()
    if norm_indicators not in ('none',''):
        ind_cols = [c for c in ['MA20','MA200','MACD','MACD_SIGNAL','MACD_HIST','Return','MA20_D1','MA20_D2','MA20_D3'] if c in df]
        df = normalize_dataframe(df, ind_cols, norm_indicators)

    # Determine which subplots are needed
    has_macd = any(x in inds for x in ['MACD','MACD_SIGNAL','MACD_HIST']) and any(c in df for c in ['MACD','MACD_SIGNAL','MACD_HIST'])
    has_returns = (('RETURN' in inds) or any(x in inds for x in ['MA20_D1','MA20_D2','MA20_D3'])) and (('Return' in df) or any(c in df for c in ['MA20_D1','MA20_D2','MA20_D3']))

    # Optionally hide volume panel if all zeros
    vol_all_zero = ('Volume' in df and df['Volume'].abs().sum() == 0)
    show_volume = not (hide_volume_if_zero and vol_all_zero)

    # Compose row order and heights
    weights = (panel_weights or DEFAULT_PANEL_WEIGHTS).copy()
    rows = []  # tuples of (name, height)
    rows.append(('price', weights.get('price', 0.6)))
    if has_macd:
        rows.append(('macd', weights.get('macd', 0.2)))
    if show_volume and 'Volume' in df:
        rows.append(('volume', weights.get('volume', 0.2)))
    if has_returns:
        rows.append(('returns', weights.get('returns', 0.2)))

    # Normalize heights to sum to 1
    total = sum(h for _, h in rows)
    row_heights = [h/total for _, h in rows]

    fig = make_subplots(
        rows=len(rows),
        cols=1,
        shared_xaxes=True,
        row_heights=row_heights,
        vertical_spacing=0.03,
        specs=[[{"secondary_y": False}] for _ in range(len(rows))]
    )

    # Map row names to indices (1-based for plotly)
    row_index = {name: i+1 for i, (name, _) in enumerate(rows)}

    # Price candles
    fig.add_trace(
        go.Candlestick(x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'], name='OHLC'),
        row=row_index['price'], col=1
    )

    # Overlay moving averages on price
    if 'MA20' in inds and 'MA20' in df:
        fig.add_trace(go.Scatter(x=df.index, y=df['MA20'], mode='lines', name='MA20', line=dict(color='blue')), row=row_index['price'], col=1)
    if 'MA200' in inds and 'MA200' in df:
        fig.add_trace(go.Scatter(x=df.index, y=df['MA200'], mode='lines', name='MA200', line=dict(color='purple')), row=row_index['price'], col=1)

    # MACD subplot
    if has_macd:
        r = row_index['macd']
        if 'MACD' in inds and 'MACD' in df:
            fig.add_trace(go.Scatter(x=df.index, y=df['MACD'], mode='lines', name='MACD', line=dict(color='teal')), row=r, col=1)
        if 'MACD_SIGNAL' in inds and 'MACD_SIGNAL' in df:
            fig.add_trace(go.Scatter(x=df.index, y=df['MACD_SIGNAL'], mode='lines', name='Signal', line=dict(color='orange')), row=r, col=1)
        if 'MACD_HIST' in inds and 'MACD_HIST' in df:
            fig.add_trace(go.Bar(x=df.index, y=df['MACD_HIST'], name='Hist', marker_color='gray'), row=r, col=1)

    # Volume subplot
    if show_volume and 'Volume' in df:
        r = row_index['volume']
        fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume', marker_color='lightgray'), row=r, col=1)

    # Returns/Derivatives subplot
    if has_returns:
        r = row_index['returns']
        if 'Return' in df:
            fig.add_trace(go.Scatter(x=df.index, y=df['Return'], mode='lines', name='Return', line=dict(color='darkgreen')), row=r, col=1)
        for cname, color in [('MA20_D1', 'red'), ('MA20_D2', 'brown'), ('MA20_D3', 'black')]:
            if cname in df:
                fig.add_trace(go.Scatter(x=df.index, y=df[cname], mode='lines', name=cname, line=dict(color=color)), row=r, col=1)

    # Trade markers on price
    def add_markers(sub, name, color, symbol):
        if not sub.empty:
            fig.add_trace(
                go.Scatter(
                    x=sub['exec_time'], y=sub['exec_price'], mode='markers', name=name,
                    marker=dict(color=color, symbol=symbol, size=9, line=dict(width=1,color='black'))
                ),
                row=row_index['price'], col=1
            )

    add_markers(aligned[aligned['action']=='OPEN_LONG'], 'Open Long', 'green', 'triangle-up')
    add_markers(aligned[aligned['action']=='OPEN_SHORT'], 'Open Short', 'orange', 'triangle-down')
    add_markers(aligned[aligned['action']=='CLOSE_LONG'], 'Close Long', 'blue', 'circle')
    add_markers(aligned[aligned['action']=='CLOSE_SHORT'], 'Close Short', 'red', 'x')

    title_suffix = ''
    if norm_ohlc not in ('none','') or norm_indicators not in ('none',''):
        title_suffix = f" • normOHLC={norm_ohlc} • normInd={norm_indicators}"

    fig.update_layout(title=f"Episode {episode} Trades{title_suffix}", xaxis_rangeslider_visible=False, height=950, template='plotly_white', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
    # Use spike lines compatible across Plotly versions
    fig.update_xaxes(showgrid=True, showspikes=True, spikemode='across', spikesnap='cursor')
    fig.update_yaxes(showgrid=True, showspikes=True, spikemode='across')

    return fig


In [10]:
# 11) Save Figures and CSV Exports

# Save enriched trades and optionally an interactive HTML for Plotly

def save_outputs(aligned: pd.DataFrame, metrics: pd.DataFrame, episode: int, fig: "go.Figure" = None, save_html: bool = False) -> Path:
    out_dir = OUTPUTS_DIR / f"episode_{episode:03d}"
    out_dir.mkdir(parents=True, exist_ok=True)
    # Save metrics
    metrics_path = out_dir / 'trades_enriched.csv'
    metrics.to_csv(metrics_path, index=False)
    print(f"Saved: {metrics_path}")
    if save_html and fig is not None:
        html_path = out_dir / 'trades.html'
        fig.write_html(str(html_path), include_plotlyjs='cdn', auto_open=False)
        print(f"Saved: {html_path}")
    return out_dir


In [12]:
# 12) Script Entrypoint Cell for VS Code (Plotly only)
if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--episode', type=int, default=None, help='Episode number to render')
    parser.add_argument('--tolerance_hours', type=float, default=2.0, help='Merge tolerance in hours')
    parser.add_argument('--save_html', action='store_true', help='Also save interactive HTML figure')
    args, _ = parser.parse_known_args()

    ep = args.episode or (int(EP_IDX['episode'].max()) if not EP_IDX.empty else None)
    if ep is None:
        print('No episodes found to render.')
    else:
        tr_raw = load_episode_trace(ep, EP_IDX)
        aligned = align_trades_to_candles(tr_raw, CANDLES, tolerance=f"{args.tolerance_hours}H")
        metrics = compute_trade_metrics(aligned)
        # Choose a sensible window
        if aligned['exec_time'].notna().any():
            tmin, tmax = aligned['exec_time'].min(), aligned['exec_time'].max()
            pad = pd.Timedelta('72H')
            df = CANDLES.loc[str(tmin - pad): str(tmax + pad)]
        else:
            df = CANDLES.tail(5000)
        fig = plot_plotly(aligned, df, ep)
        if fig is not None:
            fig.show()
        save_outputs(aligned, metrics, ep, fig=fig, save_html=args.save_html)
        print(f"Rendered episode {ep} (plotly)")


Saved: /Users/dirkiemyburgh/Documents/Courses/Binary_trading_classifier/scripts/outputs/episode_027/trades_enriched.csv
Rendered episode 27 (plotly)


In [ ]:
5000/24

208.33333333333334